# Multiple object tracker

Пример использовния трекера в связке с детектором



###Download MOT17 data and labels

In [ ]:
# Full data splitted by frames
# Really  we don't need it for this task but some images will be used for tests
#!wget https://motchallenge.net/data/MOT17Det.zip # full train

!wget https://ml.gan4x4.ru/hse/cv/MOT17-09.zip
!unzip MOT17-09.zip

# File with wideo
# Warning video resolution in this copy smaller than original video
!wget https://motchallenge.net/sequenceVideos/MOT17-09-SDP-raw.webm


In [ ]:
from IPython.display import Video

in_video = "/content/MOT17-09-SDP-raw.webm"
Video(in_video, embed=True, width=960, height=540)

### Look at GT labels

All frame numbers, target IDs and bounding boxes are 1-based. World coordinates x,y,z are ignored for the 2D challenge

Номера кадров, идентификаторы треков и bbox нумеруются с единицы. в 2D-задаче ukj,fkmyst координаты (x, y, z) игнорируются .


In [ ]:
import pandas as pd
#https://github.com/dendorferpatrick/MOTChallengeEvalKit/tree/master/MOT
gt = '/content/MOT17-09/gt/gt.txt'
labels = pd.read_csv(gt,
                     sep=',',
                     names=["frame", "id", "bb_left", "bb_top", "bb_width", "bb_height", "conf", "x", "y", "z"])

print(labels.iloc[[2]])
labels.head()

Выведем GT bbox поверх первого кадра

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

def show(n, labels):
  font = cv2.FONT_HERSHEY_SIMPLEX
  font_scale = 0.7
  font_thickness = 2
  text_color = (0, 255, 0) # Green
  filename = f"/content/MOT17-09/img1/{n:06d}.jpg"
  img = cv2.imread(filename)
  assert img is not None, f"Image {filename} not found"

  bbox_for_current_frame = labels[labels['frame'] == n]
  for _, gt_line in bbox_for_current_frame.iterrows():

    # Extract bounding box coordinates from the gt_line
    x, y, w, h = int(gt_line['bb_left']), int(gt_line['bb_top']), int(gt_line['bb_width']), int(gt_line['bb_height'])
    track_id = int(gt_line['id'])

    # Draw the bounding box in green color (BGR format)
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2) # Green color, 2 pixels thick

    # Draw the track ID
    cv2.putText(img, f"ID: {track_id}", (x, y - 10), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

  return img


# The first argument frame number (n) starting from 1 , and the second labels dataframe
img_with_bbox = show(1,labels)
cv2_imshow(img_with_bbox)

### Person detector (YOLO -based)

Класс PersonDetector.

На вход метод __call__ получает изображение(кадр) на выходе список bbox и уверенностей в формате совместимым с выбранным трекером.
Возвращаются bbox только для класса Person - остальные фильтруются.
В качестве детектора используется YOLO26

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load a COCO-pretrained YOLO26n model
model = YOLO("yolo26n.pt")

In [ ]:
first_frame_path = "/content/MOT17-09/img1/000001.jpg"
results = model.predict(first_frame_path, verbose = False)

In [ ]:
print(results[0].show())

In [ ]:
import torch

class PersonDetector:
  def __init__(self):
    # Place your code here
    self.model = YOLO("yolo26n.pt")

  def __call__(self,imgs):
    results = self.model.predict(imgs, verbose=False)
    out = []
    for r in results:
      for i, conf in enumerate(r.boxes.conf):
        if r.boxes.cls[i] == 0 : #person
          #xyxyn normalized by image size
          out.append(torch.cat((r.boxes.xyxy[i], conf.unsqueeze(0), r.boxes.cls[i].unsqueeze(0)),dim=0 ))

    return out

### Smoke test for PersonDetector

In [ ]:
pd = PersonDetector()
out = pd(first_frame_path)
print(out)

In [ ]:
detector = PersonDetector()
imgs = ['/content/MOT17-09/img1/000001.jpg','/content/MOT17-09/img1/000002.jpg']
bboxes_with_persons = detector(imgs)
print("Persons",bboxes_with_persons)


### Tracking pipeline






In [ ]:
import torch
from types import SimpleNamespace
from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker
import os # Import os module for file checks
from tqdm import tqdm


# minimal args used by BYTETracker (Ultralytics)
args = SimpleNamespace(
    track_high_thresh=0.5, # first main pass th. err: high - lost, low -swithc
    track_low_thresh=0.1, # Det. with track_low_thresh <= score < track_high_thresh are used in a second association pass
    new_track_thresh=0.5, # Minimum confidence for initializing a brand-new track
    track_buffer=30, # How many frames a track can remain “lost” (unmatched) before it’s removed
    match_thresh=0.8, # IoU (or distance) threshold for matching
    fuse_score=True, # When True, high-confidence detections are favored in matching
)

tracker = BYTETracker(args, frame_rate=30)

# Open the video
cap = cv2.VideoCapture(in_video)
H, W = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Video properties: H={H}, W={W}, FPS={fps}")

# Open output file
fourcc = cv2.VideoWriter_fourcc(*'VP90')
out = cv2.VideoWriter('output.webm', fourcc, fps, (W,H))

MAX_FRAMES = 200 # Increase frames to process more of the video
for i in tqdm(range(MAX_FRAMES)):
  ret, frame = cap.read()
  if ret:
    dets_tensor = torch.stack(pd(frame)) # [x1,y1,x2,y2,conf,cls]
    boxes = Boxes(dets_tensor , orig_shape=(H, W))
    tracks = tracker.update(boxes)  # numpy array Nx8: [x1,y1,x2,y2,id,score,cls,det_idx]
    for x1,y1,x2,y2,tid,conf,cls,det_i in tracks:
      #  # draw bbox & tid on frame
      cv2.rectangle(frame, (int(x1),int(y1)), (int(x2),int(y2)),color=(0, 255, 0), thickness=3)
      cv2.putText(frame, f"ID: {tid}", (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, fontScale = 0.7 ,color = (0, 255, 0))
    out.write(frame)

cap.release()
out.release()


Посмотрим что получилось:

In [ ]:
from IPython.display import Video


Video("/content/output.webm", embed=True, width=W, height=H)

Более простой (но менее понятный) способ использовать трекер:

In [ ]:
results = model.track("/content/MOT17-09-SDP-raw.webm", show=True, tracker="bytetrack.yaml")  # with ByteTrack